# Evaluate Step-DPO Model (Side-by-Side Comparison)

This notebook runs the base model and the DPO-finetuned model on the same holdout reasoning questions to see if the reasoning path improved.

In [ ]:
import json
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from qwen_vl_utils import process_vision_info

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
adapter_path = "qwen_vl_step_dpo_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)

# Load fine-tuned model (base + LoRA)
ft_model = PeftModel.from_pretrained(base_model, adapter_path)

In [ ]:
def generate_response(model, processor, image_path, question):
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": "Analyze this chart. Provide step-by-step reasoning and a final answer.\n" + question}
        ]}
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")
    
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=512, temperature=0.0)
        
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text[0]

# Sample holdout questions (you can load eval_reasoning_ids.json here)
test_samples = [
    {"id": "123", "image_path": "data/CharXiv/images/123.jpg", "question": "What is the value of X?"}
]

for sample in test_samples:
    print(f"--- Question {sample['id']} ---")
    print(f"Q: {sample['question']}\n")
    
    with base_model.disable_adapter():
        base_out = generate_response(base_model, processor, sample['image_path'], sample['question'])
    
    ft_out = generate_response(ft_model, processor, sample['image_path'], sample['question'])
    
    print(f"[BASE MODEL]:\n{base_out}\n")
    print(f"[FINE-TUNED MODEL]:\n{ft_out}\n")
    print("===========================================\n")